# You Already Know Neural Networks

This lecture borrows heavily from Chapter 9 of HOML.

**Spoiler alert!** You've been training simple neural networks for much of the semester.

This is the first of three lectures connecting the supervised ML foundations from this course to neural networks and deep learning. The thesis: everything in a neural network is something you've already seen under a different name. Today we assemble the pieces.

The progression that brought us here:

| Step | Name | Formulation |
|---|---|---|
| 1 | Linear equation | $y = mx + b$ |
| 2 | Simple Linear Regression (SLR) | $\hat{y} = w_1 x + b$ |
| 3 | Multiple Linear Regression (MLR) | $z = w_1 x_1 + w_2 x_2 + \ldots + w_p x_p + b$ |
| 4 | Logistic Regression + gradient descent | $\hat{y} = \sigma(z)$ |

That progression leads us directly to neural networks. You have every prerequisite: linear combinations of inputs, nonlinear transformations, loss functions, optimization, regularization, cross-validation. You just haven't put them together yet.

The name is intimidating and the details are not trivial, but these final lectures will give you a basic understanding and intuition of NNs, connect them to what you've learned so far, and give you a strong basis for further study in the "modern miracle" of Deep Learning.

## Setup

We're going to continue using sklearn today, but will move to another framework for the next lecture.

In [ ]:
%matplotlib inline

import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml, make_moons
from sklearn.dummy import DummyClassifier
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

---

## Part 1: From Neurons to Networks

### Biological inspiration

A biological neuron receives electrical signals from other neurons through its dendrites. If the combined signal exceeds a threshold, the neuron fires - sending its own signal through the axon to other neurons. The brain contains roughly 86 billion of these, connected in a vast network.

![](images/15a-homl9-1-neuron.png)

In the 1940s, McCulloch and Pitts proposed a simplified mathematical model. Their *artificial neuron* has one or more binary (on/off) inputs and one binary output. The artificial neuron activates its output when more than a certain number of its inputs are active. It was the starting point for everything that follows.

![](images/15a-homl9-3-ann.png)

Note that some dispute the biological inspiration or relevance. It's less important to try to model the brain exactly than it is to be inspired by it. This practice of drawing design ideas from nature is called *biomimicry* - airplane wings inspired by bird wings, Velcro inspired by burrs, sonar inspired by bats.

### The Perceptron

In 1958, Frank Rosenblatt took this idea and made it learnable. His Perceptron adjusts its weights based on errors - when it misclassifies an input, it nudges the weights in the right direction. This is remarkably similar to gradient descent.

The core unit is a *Threshold Logic Unit* (TLU): it computes a weighted sum of its inputs plus a bias term, then applies a step function. If the sum is positive, output 1; otherwise, output 0.

![](images/15a-homl9-4-tlu.png)

This should seem familiar. It's the same weighted sum from multiple linear regression (Day 6). The only difference is the output function: MLR uses the raw value, LogReg wraps it in a sigmoid, and the TLU applies a hard threshold.

$$z = w_1 x_1 + w_2 x_2 + \ldots + w_p x_p + b$$

$$\hat{y} = \text{step}(z) = \begin{cases} 1 & \text{if } z \geq 0 \\ 0 & \text{if } z < 0 \end{cases}$$


A Perceptron is a single layer of TLUs: every input connects to every output (fully connected). For binary classification, there's one output TLU. For multiclass, there's one per class. Note that the perceptron does not predict probabilities in the way that LogReg does - classes are predicted directly.

![](images/15a-homl9-5-perceptron.png)

Note that this figure does not show the weights or bias term, only the structure (*architecture*). It is a bit confusing in that regard. The computation is clearer in the previous notation and figure 9-4.

For a perceptron with $p$ inputs and $n$ TLUs, the output of TLU $j$ is:

$$\hat{y}_j = \text{step}\left(\sum_{i=1}^{p} w_{ij} x_i + b_j\right) \quad \text{for } j = 1, 2, \ldots, n$$

Every output is a linear combination of the inputs passed through a step function. No matter how many TLUs you stack side-by-side in this single layer, each one can only learn a linear boundary. The perceptron is fundamentally linear.

### The Perceptron in sklearn

sklearn has a `Perceptron` class. Let's see it on a simple 2D problem.

In [ ]:
# A linearly separable problem
from sklearn.datasets import make_blobs

# make_blobs creates easily seperable classes
X_blobs, y_blobs = make_blobs(
    n_samples=200, centers=2, n_features=2, random_state=42
)

perceptron = make_pipeline(StandardScaler(), Perceptron(random_state=42))
perceptron.fit(X_blobs, y_blobs)

fig, ax = plt.subplots(figsize=(6, 4))
DecisionBoundaryDisplay.from_estimator(
    perceptron, X_blobs, ax=ax, cmap="coolwarm", alpha=0.3, response_method="predict"
)
ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap="coolwarm",
           s=30, edgecolors="k", linewidths=0.5)
acc = perceptron.score(X_blobs, y_blobs)
ax.set_title(f"Perceptron on Linearly Separable Data\nAccuracy: {acc:.3f}")
plt.tight_layout()
plt.show()

The perceptron works perfectly here because the data *is* linearly separable - one straight line divides the two classes. The perceptron finds that line by iteratively adjusting weights, much like gradient descent.

But what about data that isn't linearly separable?

In [ ]:
# A non-linear problem: make_moons
X_moons, y_moons = make_moons(n_samples=500, noise=0.2, random_state=42)

perceptron_moons = make_pipeline(StandardScaler(), Perceptron(random_state=42))
perceptron_moons.fit(X_moons, y_moons)

fig, ax = plt.subplots(figsize=(6, 4))
DecisionBoundaryDisplay.from_estimator(
    perceptron_moons, X_moons, ax=ax, cmap="coolwarm", alpha=0.3, response_method="predict"
)
ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="coolwarm",
           s=20, edgecolors="k", linewidths=0.3)
acc = perceptron_moons.score(X_moons, y_moons)
ax.set_title(f"Perceptron on make_moons\nAccuracy: {acc:.3f}")
plt.tight_layout()
plt.show()

The perceptron draws a straight line through data that needs a curve. It can't do better - a single layer of TLUs can only learn *linear* decision boundaries.

### Minsky's critique and the AI winter

The perceptron's limitation isn't surprising - of course a linear classifier can't learn nonlinear boundaries. What mattered was context.

Rosenblatt and the perceptron community had made extravagant claims. The New York Times ran a 1958 headline about the Navy building a machine that would eventually "walk, talk, see, write, reproduce itself and be conscious of its existence." The gap between the hype and what a linear classifier can actually do was enormous.

In 1969, Marvin Minsky and Seymour Papert published *Perceptrons*, proving the limitation rigorously. Their contribution wasn't the mathematics - it was the argument that there was no practical path forward. Multi-layer networks could theoretically overcome the limitation, but no one knew how to *train* them effectively. The training problem was genuinely unsolved.

Minsky was the most influential voice in AI. The book gave funding agencies an excuse to redirect resources away from neural networks. Research dried up for over a decade - the first "AI winter." It wasn't a mathematical event; it was a sociological one.

The limitation was real. The training problem was real. The conclusion that neural networks were a dead end was wrong.

Marvin Minsky was a co-founder of the MIT AI Lab and one of the most influential voices in AI research for decades. He won the Turing Award in 1969 - the same year *Perceptrons* was published. Seymour Papert, his co-author, was also at MIT and is better known today for developing the Logo programming language and for constructionist learning theory. When these two argued that a research direction was a dead end, funding agencies listened.

### The Multi-Layer Perceptron

The *multi-layer perceptron* (MLP) adds one or more *hidden layers* between inputs and outputs. Each hidden neuron computes a weighted sum of its inputs and applies an activation function.

The internal layers are said to be hidden because their activations aren't directly visible. You provide $X$ (input layer) and $y$ (output layer), the network does the rest during the training process. This is what makes neural networks powerful: feature engineering is automated as the hidden layers learn useful representations directly from the data.

![](images/15a-homl9-7-mlp.png)

The hidden neurons function just like TLUs, but with a critical difference. In order to train this network - to calculate the weights and biases through an algorithm like gradient descent - we need to be able to relate the error of a prediction back to the associated weights. We need the partial derivatives.

But the step function in the original perceptron is not differentiable, it jumps from 0 to 1 with no smooth transition. This makes gradient-based training impossible for multiple layers, because you can't compute "how much did this weight contribute to the error?" through a discontinuous function.

The solution: replace the step function with a smooth, differentiable activation. The sigmoid was the original choice (the same $\sigma$ from logistic regression). With differentiable activations, you can use the chain rule to propagate error gradients backwards through the network - the *backpropagation* algorithm (Rumelhart, Hinton & Williams, 1986). We'll cover backprop in detail in Lecture 2.

### Why activations matter

A key insight: if every neuron used a *linear* activation function (i.e., just passed through the weighted sum unchanged), then stacking layers would be pointless. A linear function of a linear function is still linear. No matter how many layers you add, the whole network would collapse to a single linear transformation - no better than logistic regression.

Non-linear activations are what make depth useful. Each layer applies a non-linear transformation, and the composition of many non-linear transformations can represent arbitrarily complex functions.

The sigmoid ($\sigma$) from logistic regression was the original activation choice: smooth, differentiable, squashes any input to (0, 1). It works, but causes problems in deep networks - we'll see why in Lecture 2. The modern default is the humble *ReLU* (Rectified Linear Unit):

$$\text{ReLU}(z) = \max(0, z)$$

In [ ]:
z = np.linspace(-3, 3, 200)
step = np.where(z >= 0, 1, 0)
relu = np.maximum(0, z)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))

# Step function (perceptron)
axes[0].plot(z, step, linewidth=2, color="tomato", drawstyle="steps-post")
axes[0].axhline(0, color="gray", linewidth=0.5)
axes[0].axvline(0, color="gray", linewidth=0.5)
axes[0].set_xlabel("input ($z$)")
axes[0].set_ylabel("output")
axes[0].set_title("Step (original perceptron)")
axes[0].grid(True, alpha=0.3)

# ReLU
axes[1].plot(z, relu, linewidth=2, color="steelblue")
axes[1].axhline(0, color="gray", linewidth=0.5)
axes[1].axvline(0, color="gray", linewidth=0.5)
axes[1].set_xlabel("input ($z$)")
axes[1].set_ylabel("output")
axes[1].set_title("ReLU (modern default)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Simple: if the input is positive, pass it through; if negative, output zero. Despite its simplicity, ReLU is what made training deep networks practical. It's the default in `MLPClassifier` (`activation='relu'`).

This is the *Universal Approximation Theorem* (Cybenko 1989, Hornik 1991): a single hidden layer with enough neurons and a non-linear activation can approximate any continuous function to arbitrary precision.

This is an *existence* theorem, not a *construction* theorem. It says a solution exists. It doesn't tell you how many neurons you need, how long training will take, or whether gradient descent will find it.

Still, take a moment to reflect on its implications...

A weighted sum of polynomial terms ($x$, $x^2$, $x^3$, ...) can approximate any continuous function, given enough terms. Hidden layers do something similar but the network learns equivalent features during training. You don't have to specify the polynomial expansion, test various model complexities, consider interaction effects, etc. The best representation of the data is *learned* by the network.

Cybenko's theorem says one hidden layer is *sufficient*. In practice, deeper networks with fewer neurons per layer work much better - understanding why is the Lecture 2 story.

### The modern MLP

A modern MLP for classification has:

- An *input layer*: one neuron per feature (not learned, just passes data through)
- One or more *hidden layers*: each neuron computes a weighted sum + activation
- An *output layer*: one neuron per class, with *softmax* activation that converts raw scores to probabilities summing to 1

Recall that softmax is the multiclass generalization of the sigmoid from logistic regression:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

where $z_i$ is the weighted sum from MLR.

> Binary logistic regression from Day 14 is a single-neuron MLP with sigmoid activation. Multiclass logistic regression is a K-neuron output layer with softmax. Either way, you've been training a neural network all semester.

NNs compose such neurons into arbitrary *architectures*. The number of layers, neurons per layer, activation functions, and loss function are what you design. Everything else is learned.

| Component | What you choose | What the network learns |
|---|---|---|
| Number of hidden layers | Architecture decision | - |
| Neurons per layer | Architecture decision | - |
| Activation function | Architecture decision | - |
| Output function | Determined by problem type (softmax / sigmoid / identity) | - |
| Loss function | Architecture decision | - |
| Weights and biases | - | Learned via gradient descent |
| Features in each layer | - | Learned automatically |

![](images/15a-homl9-10-mlp-class.png)

### Vocabulary mapping

Here's the connection between what you already know and neural network terminology:

| What you called it | What neural networks call it |
|---|---|
| Coefficients ($w_1, w_2, \ldots$) | Weights |
| Intercept ($b$) | Bias |
| Sigmoid ($\sigma$) | Activation function (one of many) |
| Log-loss / cross-entropy | Loss function |
| Gradient descent | Optimizer |
| Preprocessing (scaling, encoding) | Part of the pipeline (same) |
| Feature engineering (polynomials, interactions) | What hidden layers do automatically |

### Recommended viewing

For an excellent visual walkthrough - the perceptron, Minsky's critique, and how stacking neurons solves it - see Welch Labs: [*ChatGPT is made from 100 million of these*](https://www.youtube.com/watch?v=l-9ALe3U-Fg) (~24 min). It builds a 3-neuron network from scratch and shows exactly how two hidden neurons learn two linear boundaries that a single neuron cannot.

---

## Part 2: MLPClassifier on Familiar Data

### Where the perceptron fails, the MLP succeeds

The make_moons dataset defeated the perceptron. Let's see what an MLP with a hidden layer can do.

In [ ]:
# Three approaches to nonlinear classification
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(random_state=42)
    ),
    "MLP (1 hidden layer)": make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(100,), max_iter=2000,
            random_state=42,
        ),
    ),
    "SVM (rbf kernel)": make_pipeline(
        StandardScaler(), SVC(kernel="rbf", random_state=42)
    ),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_moons, y_moons)
    DecisionBoundaryDisplay.from_estimator(
        model, X_moons, ax=ax, cmap="coolwarm", alpha=0.3, response_method="predict"
    )
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="coolwarm",
               s=20, edgecolors="k", linewidths=0.3)
    score = model.score(X_moons, y_moons)
    ax.set_title(f"{name}\nAccuracy: {score:.3f}")

plt.suptitle("Three Approaches to Nonlinear Classification", y=1.02)
plt.tight_layout()
plt.show()

LogReg draws a straight line - same as the perceptron, because it's the same computation with a different output function. The MLP with a hidden layer learns a curved boundary that follows the moon shapes. SVM(rbf) achieves a similar result.

Three approaches to the same problem:

- *LogReg*: no transformation. Linear boundary only.
- *MLP*: *learns* a transformation. The hidden layer remaps the data into a space where it becomes linearly separable.
- *SVM(rbf)*: uses a *predefined* transformation. The kernel maps to a higher-dimensional space where a linear boundary works.

The MLP discovers what it needs. The SVM uses a fixed recipe (the rbf kernel). This is the whole story of deep learning in miniature: replacing hand-designed transformations with learned ones.

### What the hidden layer actually learns

Each hidden neuron produces an activation for every input point.

We can extract these to see the learned feature space.

In [ ]:
# Train an MLP and extract hidden layer activations
mlp_pipe = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(100,), max_iter=2000,
        random_state=42,
    ),
)
mlp_pipe.fit(X_moons, y_moons)

# Access internals
mlp = mlp_pipe.named_steps["mlpclassifier"]
scaler = mlp_pipe.named_steps["standardscaler"]
X_scaled = scaler.transform(X_moons)

# Forward pass to hidden layer
hidden_input = X_scaled @ mlp.coefs_[0] + mlp.intercepts_[0]
hidden_activations = np.maximum(0, hidden_input)  # relu activation

# Show 3 learned features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, ax in enumerate(axes):
    scatter = ax.scatter(
        X_moons[:, 0], X_moons[:, 1],
        c=hidden_activations[:, i], cmap="viridis",
        s=20, alpha=0.7, edgecolors="none",
    )
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_title(f"Hidden Neuron {i+1} Activation")
    plt.colorbar(scatter, ax=ax)
plt.suptitle("What the Hidden Layer Sees", y=1.02)
plt.tight_layout()
plt.show()

Interpretation:

- each plot is for one neuron
- each dot is a training observation
- color shows that neuron's activation value (output) for that observation; yellow = strong, purple = none
- the boundary between zero and non-zero comes from the weights and biases learned by the neuron (linear)
- the activation strength is based on how far each point is from that boundary, min zero (ReLU)

The model outputs are computed from the neuron activations, NOT the original training data. Each neuron's activation is a learned feature, and each feature responds to a different region of the input space. 

This is feature *construction*, not feature *selection*. The neuron isn't choosing from existing features - it's creating new ones by learning which linear combinations of inputs are useful.

Decision trees also learn features implicitly: each split creates a binary feature ("is pixel 392 > 128?"). The difference: trees learn axis-aligned thresholds on individual features; neural networks learn continuous weighted combinations across *all* features simultaneously.

In 2D (make_moons), the learned features are spatial regions. In high-dimensional data like MNIST, the first hidden layer learns patterns that respond to combinations of pixel intensities - we'll visualize this in Part 3. This is why you will often hear that NNs learn *representations* or *abstractions* of the data.

Hidden neurons don't directly predict classes or values; they are task agnostic. The features they construct are used by the output layer to do so. For classification problems the scores are interpreted as probabilities using sigmoid / softmax. For regression, they are used directly.

A note on scaling: neural networks require scaled inputs for the same reason LogReg does. The weights are initialized from a small random distribution, and gradient descent assumes all features contribute on comparable scales. Unscaled features with large magnitudes dominate the gradient, causing slow or unstable convergence. In sklearn, the cleanest way is to wrap `MLPClassifier` / `MLPRegressor` in a `Pipeline` with `StandardScaler`. Some data has a natural range (MNIST pixels are bounded to `[0, 255]`, so dividing by 255 works fine) - the important thing is that features enter the network on comparable scales.

### The sklearn API you already know

`MLPClassifier` works exactly like every other sklearn estimator. Same `fit`/`predict`/`score`. Same `cross_val_score`. Same `Pipeline`.

In [ ]:
cv_models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(random_state=42)
    ),
    "KNN (k=5)": make_pipeline(
        StandardScaler(), KNeighborsClassifier()
    ),
    "SVM (rbf)": make_pipeline(
        StandardScaler(), SVC(random_state=42)
    ),
    "MLP (100 neurons)": make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(100,), max_iter=2000,
            random_state=42,
        ),
    ),
    "MLP (2 layers)": make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(100, 50), max_iter=2000,
            random_state=42,
        ),
    ),
}

print(f"{'Model':<25s}  {'CV Accuracy (mean)':>18s}  {'(std)':>7s}")
print("-" * 55)
for name, model in cv_models.items():
    cv = cross_val_score(model, X_moons, y_moons, cv=5, scoring="accuracy")
    print(f"{name:<25s}  {cv.mean():>18.4f}  {cv.std():>7.4f}")

---

## Part 3: MNIST - A Better Test

### The dataset you know

MNIST was the workhorse of 09b - 70,000 handwritten digits, 784 features (28×28 pixels). You classified these with LogReg, KNN, and SVM. Now let's see what a neural network does.

In [ ]:
# Load MNIST
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
X_mnist, y_mnist = mnist.data, mnist.target.astype(int)

# Same split as 09b
X_train, X_test = X_mnist[:60000], X_mnist[60000:]
y_train, y_test = y_mnist[:60000], y_mnist[60000:]

# Scale pixels to [0, 1]
X_train_scaled = X_train / 255.0
X_test_scaled = X_test / 255.0

print(f"Training: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
print(f"Test:     {X_test.shape[0]:,} samples")

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(14, 1.5))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i].reshape(28, 28), cmap="gray")
    ax.set_title(str(y_train[i]), fontsize=10)
    ax.axis("off")
plt.suptitle("First 10 MNIST Training Images", y=1.05)
plt.tight_layout()
plt.show()

### Head-to-head comparison

In [ ]:
mnist_models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "KNN (k=5)": KNeighborsClassifier(),
    "MLP (128, 64)": MLPClassifier(
        hidden_layer_sizes=(128, 64), max_iter=100,
        early_stopping=True, random_state=42,
    ),
}

results = []
for name, model in mnist_models.items():
    start = time.time()
    model.fit(X_train_scaled, y_train)
    train_time = time.time() - start

    test_acc = model.score(X_test_scaled, y_test)
    results.append({
        "Model": name,
        "Test Accuracy": test_acc,
        "Train Time (s)": train_time,
    })

results_df = pd.DataFrame(results)
results_df["Test Accuracy"] = results_df["Test Accuracy"].map("{:.4f}".format)
results_df["Train Time (s)"] = results_df["Train Time (s)"].map("{:.1f}".format)
print(results_df.to_string(index=False))

The MLP beats LogReg and KNN. LogReg can only learn linear relationships between pixel intensities and digit classes. KNN works well because similar digits have similar pixel patterns, but it's slow at prediction time. The MLP learns nonlinear features - combinations of pixels - that are more informative than raw pixel values.

### What the hidden neurons look for

In Part 2 we saw that hidden neurons on make_moons learn to respond to different spatial regions. On MNIST, with 784 pixel inputs, we can visualize what each neuron has learned by reshaping its 784 weights back to 28×28.

In [ ]:
# Train an MLP and visualize first-layer weight vectors
mlp_viz = MLPClassifier(
    hidden_layer_sizes=(128, 64), max_iter=100,
    early_stopping=True, random_state=42,
)
mlp_viz.fit(X_train_scaled, y_train)

# Each column of coefs_[0] is a 784-dim weight vector for one hidden neuron
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.ravel()):
    weights = mlp_viz.coefs_[0][:, i].reshape(28, 28)
    ax.imshow(weights, cmap="RdBu_r", interpolation="nearest")
    ax.set_title(f"N{i+1}", fontsize=8)
    ax.axis("off")
plt.suptitle("First 16 Hidden Neurons — What They Look For in MNIST", y=1.02)
plt.tight_layout()
plt.show()

Bright regions are pixel positions the neuron responds positively to; dark regions are positions it responds negatively to. Some neurons look like blurry digit templates. Others respond to strokes or curves in specific positions. The network has discovered features that help distinguish digits - without anyone telling it what a digit looks like.

### The training loss curve

`MLPClassifier` tracks the loss at each iteration. This is the same learning curve you've been reading since Day 7.

In [ ]:
mlp_track = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    max_iter=100,
    random_state=42,
)
mlp_track.fit(X_train_scaled, y_train)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mlp_track.loss_curve_, linewidth=2)
ax.set_xlabel("Iteration")
ax.set_ylabel("Loss (cross-entropy)")
ax.set_title("MLP Training Loss Curve — MNIST")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"LogReg parameters:  {784 * 10 + 10:,} (784 features × 10 classes + biases)")
print(f"MLP parameters:     {784*128 + 128 + 128*64 + 64 + 64*10 + 10:,}"
      f" (784→128→64→10 + biases)")

Same cross-entropy loss. The weights are updated using *backpropagation* - the chain rule applied layer by layer to figure out how much each weight contributed to the error - followed by gradient descent to take a step. Backprop computes *which direction* to move; gradient descent *takes the step*. We'll cover how backprop works in Lecture 2. For now, the point is: same loss function, same optimization idea, 14× more parameters.

---

## Part 4: Hyperparameters and Regularization

### Architecture

The most important choice is the network architecture - how many layers, how many neurons per layer. This controls the model's *capacity*: its ability to represent complex functions.

In [ ]:
architectures = {
    "(32,)":           (32,),
    "(128,)":          (128,),
    "(128, 64)":       (128, 64),
    "(256, 128, 64)":  (256, 128, 64),
}

print(f"{'Architecture':<22s}  {'Params':>8s}  {'Test Acc':>9s}")
print("-" * 44)
for name, layers in architectures.items():
    mlp = MLPClassifier(
        hidden_layer_sizes=layers, max_iter=100,
        early_stopping=True, random_state=42,
    )
    mlp.fit(X_train_scaled, y_train)
    test_acc = mlp.score(X_test_scaled, y_test)

    n_params = 0
    layer_sizes = [784] + list(layers) + [10]
    for i in range(len(layer_sizes) - 1):
        n_params += layer_sizes[i] * layer_sizes[i + 1] + layer_sizes[i + 1]

    print(f"{name:<22s}  {n_params:>8,}  {test_acc:>9.4f}")

The same bias-variance story applies. Too few neurons → underfitting. Too many → potential overfitting. But notice: neural networks are remarkably tolerant of being "too big." The (256, 128, 64) model may not overfit much worse than (128, 64) despite having far more parameters. We'll see why in Lecture 3 when we discuss double descent.

### Learning rate sensitivity

The learning rate controls the step size of gradient descent. Same role as `learning_rate` in gradient boosting (13b), same tradeoffs.

In [ ]:
learning_rates = [0.0001, 0.001, 0.01, 0.1]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, lr in zip(axes, learning_rates):
    mlp = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        learning_rate_init=lr,
        max_iter=100,
        random_state=42,
    )
    mlp.fit(X_train_scaled, y_train)
    ax.plot(mlp.loss_curve_, linewidth=2)
    test_acc = mlp.score(X_test_scaled, y_test)
    ax.set_title(f"lr={lr}\nTest Acc: {test_acc:.4f}", fontsize=10)
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.3)
plt.suptitle("Learning Rate Sensitivity", y=1.02)
plt.tight_layout()
plt.show()

Too low and it converges painfully slowly. Too high and it oscillates or diverges. The default (`0.001`) is usually reasonable.

### Early stopping

Neural networks can overfit. Early stopping monitors the validation loss and stops training when it starts increasing - the same idea as staged predictions in gradient boosting (13b).

In [ ]:
mlp_early = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64), max_iter=200,
    early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=10, random_state=42,
)
mlp_early.fit(X_train_scaled, y_train)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mlp_early.loss_curve_, label="Training loss", linewidth=2)
if hasattr(mlp_early, "validation_scores_"):
    ax_twin = ax.twinx()
    ax_twin.plot(mlp_early.validation_scores_, color="tomato",
                 linestyle="--", label="Validation accuracy", linewidth=1.5)
    ax_twin.set_ylabel("Validation Accuracy", color="tomato")
    ax_twin.legend(loc="center right")
ax.set_xlabel("Iteration")
ax.set_ylabel("Loss")
ax.set_title(f"Early Stopping — stopped at iteration {mlp_early.n_iter_}, "
             f"Test Acc: {mlp_early.score(X_test_scaled, y_test):.4f}")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Regularization you already know

Neural networks have their own vocabulary, but the ideas map directly to this course:

| This Course | Neural Network Name | Where You Learned It |
|---|---|---|
| L2 regularization ($\lambda \sum w_i^2$) | Weight decay (`alpha` in MLPClassifier) | 06a (Ridge) |
| Early stopping (stop when validation worsens) | Early stopping | 13b (Gradient Boosting) |
| Random feature subsets at each split | Dropout (random neuron deactivation) | 13a (Random Forests) |
| Cross-validation | Validation split | 05b |
| `learning_rate` (step size for each tree) | `learning_rate_init` | 13b (Gradient Boosting) |

The `alpha` parameter in `MLPClassifier` is L2 regularization - the same $\lambda \sum w_i^2$ penalty from Ridge regression (06a). It's called "weight decay" in the neural network literature, but the math is identical.

Dropout deserves a note. `MLPClassifier` doesn't support it, but the idea is the same as random forests' `max_features`: randomly remove information during training to force the model to be robust. In RF, you randomly exclude features at each split. In dropout, you randomly deactivate neurons during each training step. Both prevent co-adaptation. We'll see dropout in action with PyTorch in Lecture 2.

---

## Part 5: MLP in the Baseline Hierarchy

Where does the MLP sit in the progression we've been building all semester?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
names = [r["Model"] for r in results]
accs = [r["Test Accuracy"] for r in results]
times = [r["Train Time (s)"] for r in results]

colors = ["gray"] + ["steelblue"] * 2 + ["tomato"]
bars = ax.barh(names, accs, color=colors, edgecolor="black")
ax.set_xlabel("Test Accuracy")
ax.set_title("Baseline Hierarchy — MNIST")
ax.set_xlim(0, 1.05)
for bar, acc, t in zip(bars, accs, times):
    ax.text(acc + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{acc:.3f} ({t:.1f}s)", va="center", fontsize=10)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

The hierarchy tells the story of the data:

- Dummy → LogReg gap: the data has learnable structure
- LogReg → KNN gap: there's nonlinear structure that a linear model misses
- KNN → MLP gap: learned features capture the structure better than raw pixel distance

For your project, `MLPClassifier` (or `MLPRegressor` for regression) is a legitimate choice for the "learn independently" model requirement.

A note on `hidden_layer_sizes`: the tuple `(128, 64)` means two hidden layers with 128 and 64 neurons respectively. sklearn automatically adds the output layer (10 neurons for 10-class classification, or 1 for regression). The numbers are architectural choices - there's no formula to derive them. Common practice is to start with one or two hidden layers and experiment.

Practical hyperparameter guidance:

| Parameter | Default | Guidance |
|---|---|---|
| `hidden_layer_sizes` | `(100,)` | Start with one layer of 100. Add a second layer or increase width if underfitting. |
| `activation` | `'relu'` | Use relu. Lecture 2 explains why. |
| `learning_rate_init` | `0.001` | Good default for Adam optimizer. Lower (0.0001) if training is unstable. |
| `max_iter` | `200` | Increase if you see convergence warnings. |
| `early_stopping` | `False` | Set to `True` to prevent overfitting. |
| `alpha` | `0.0001` | L2 regularization (weight decay). Increase if overfitting. |
| `solver` | `'adam'` | Adam is the modern default. Use it. |

A minimal project setup:

```python
from sklearn.neural_network import MLPClassifier  # or MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(100,),
        max_iter=500,
        early_stopping=True,
        random_state=42,
    ),
)
```

Treat it like any other sklearn estimator. Same baseline hierarchy, same comparison framework, same `cross_val_score`.

---

## Part 6: What's Under the Hood

### The loss function is familiar

For classification, `MLPClassifier` minimizes cross-entropy - the same loss function as logistic regression. For regression, `MLPRegressor` minimizes squared error - the same as OLS. Nothing new.

The optimizer is also familiar: gradient descent on all weights simultaneously. The same algorithm from 08b, just applied to more parameters. `MLPClassifier` uses a variant called Adam by default, which adapts the learning rate per parameter - but the core idea is identical.

### Scale matters

`MLPClassifier` trained on MNIST in seconds. Tens of thousands of parameters, tens of thousands of samples. That's the regime where sklearn works fine.

GPT-4 has an estimated hundreds of billions of parameters and trained on trillions of tokens across tens of thousands of GPUs over several months. Same fundamental algorithm - weighted sums, activations, gradient descent. The difference is scale, and scale changes more than just the training time. It changes what the network can do.

What happens when you go from hundreds of parameters to hundreds of billions? Some surprising things. We'll get there in Lecture 3.

---

## Summary

The progression, made explicit:

$$y = mx + b \;\to\; \text{SLR} \;\to\; \text{MLR} \;\to\; \text{LogReg} \;\to\; \text{TLU / Perceptron} \;\to\; \text{MLP} \;\to\; \text{Deep Network}$$

Everything in a neural network maps to something you already know:

| Neural Network Concept | Course Equivalent |
|---|---|
| Neuron / TLU | Logistic regression (one unit) |
| Hidden layer | Learned feature engineering |
| Weights and biases | Coefficients and intercept |
| Activation function | Sigmoid (LogReg), step (Perceptron), thresholds (trees) |
| Loss function | Cross-entropy (classification), MSE (regression) |
| Optimizer | Gradient descent |
| Weight decay | L2 / Ridge regularization |
| Early stopping | Same name, same idea (Gradient Boosting) |
| Dropout | Random feature subsets (Random Forests) |
| Architecture | Model complexity (`max_depth`, `n_estimators`) |

Thursday: what makes depth work (activation functions, backpropagation), and the jump from sklearn to PyTorch.